In [1]:
import os, sys
import numpy as np
import torch
import espaloma as esp
import qcportal as ptl
from collections import Counter
from openff.toolkit.topology import Molecule
from openff.qcsubmit.results import BasicResultCollection
from simtk import unit
from simtk.unit import Quantity
from matplotlib import pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
def get_graph(mol, energy, grad):
    offmol = Molecule.from_qcschema(mol, allow_undefined_stereo=True)   # convert to OpenFF Molecule object
    g = esp.Graph(offmol)

    # energy is already hartree
    g.nodes["g"].data["u_ref"] = torch.tensor(
        [
            Quantity(
                energy,
                esp.units.HARTREE_PER_PARTICLE,
            ).value_in_unit(esp.units.ENERGY_UNIT)
        ],
        dtype=torch.get_default_dtype(),
    )[None, :]

    g.nodes["n1"].data["xyz"] = torch.tensor(
        np.stack(
            [
                Quantity(
                    mol.geometry,
                    unit.bohr,
                ).value_in_unit(esp.units.DISTANCE_UNIT)
            ],
            axis=1,
        ),
        requires_grad=True,
        dtype=torch.get_default_dtype(),
    )

    g.nodes["n1"].data["u_ref_prime"] = torch.stack(
        [
            torch.tensor(
                Quantity(
                    grad,
                    esp.units.HARTREE_PER_PARTICLE / unit.bohr,
                ).value_in_unit(esp.units.FORCE_UNIT),
                dtype=torch.get_default_dtype(),
            )
        ],
        dim=1,
    )
    
    return g

In [3]:
collection_type = "Dataset"
name = "RNA Single Point Dataset v1.0"

client = ptl.FractalClient()
collection = client.get_collection(collection_type, name)

In [4]:
recs_wb97m = collection.get_records(method='wb97m-d3bj', basis='def2-tzvppd', program='psi4', keywords='wb97m-d3bj/def2-tzvppd')

In [5]:
gs = []

for i in range(len(recs_wb97m)):
    if recs_wb97m.iloc[i].record.status == 'COMPLETE':
        print("#{}: {}".format(i, recs_wb97m.iloc[i].name))
        
        mol = client.query_molecules(recs_wb97m.iloc[i].record.molecule)[0]
        energy = recs_wb97m.iloc[i].record.properties.scf_total_energy
        grad = recs_wb97m.iloc[i].record.return_result
        
        g = get_graph(mol, energy, grad)
        gs.append(g)
        #break

Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 28, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 62, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

#506: Nc1ccn([C@@H]2O[C@H](CO)[C@@H](OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3O)[C@H]2O)c(=O)n1-14


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 28, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 62, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

#528: Nc1ccn([C@@H]2O[C@H](CO)[C@@H](OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3O)[C@H]2O)c(=O)n1-34


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 28, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 62, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

...

#4396: Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])O[C@@H]2[C@@H](COP(=O)([O-])O[C@@H]3[C@@H](CO)O[C@@H](n4cnc5c(N)ncnc54)[C@@H]3O)O[C@@H](n3cnc4c(N)ncnc43)[C@@H]2O)[C@@H](O)[C@H]1O-93
#4398: Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])O[C@@H]2[C@@H](COP(=O)([O-])O[C@@H]3[C@@H](CO)O[C@@H](n4cnc5c(N)ncnc54)[C@@H]3O)O[C@@H](n3cnc4c(N)ncnc43)[C@@H]2O)[C@@H](O)[C@H]1O-95


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 31, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 33, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 34, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 64, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 63, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 66, aromatic: False, chiral: False
bo

#4399: Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])O[C@@H]2[C@@H](COP(=O)([O-])O[C@@H]3[C@@H](CO)O[C@@H](n4cnc5c(N)ncnc54)[C@@H]3O)O[C@@H](n3cnc4c(N)ncnc43)[C@@H]2O)[C@@H](O)[C@H]1O-96


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 31, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 33, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 34, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 64, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 63, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 66, aromatic: False, chiral: False
bo

#4401: Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])O[C@@H]2[C@@H](COP(=O)([O-])O[C@@H]3[C@@H](CO)O[C@@H](n4cnc5c(N)ncnc54)[C@@H]3O)O[C@@H](n3cnc4c(N)ncnc43)[C@@H]2O)[C@@H](O)[C@H]1O-98


In [6]:
ds = esp.data.dataset.GraphDataset(gs)

In [7]:
len(ds)

1564

In [8]:
ds.shuffle(seed=2666)
#ds_tr, ds_vl, ds_te = ds.split([8, 1, 1])
ds_tr, ds_vl, ds_te = ds.split([1, 1, 8])

In [9]:
representation = esp.nn.Sequential(
    layer=esp.nn.layers.dgl_legacy.gn("SAGEConv"),   # use SAGEConv implementation in DGL
    config=[128, "relu", 128, "relu", 128, "relu"],  # 3 layers, 128 units, ReLU activation
)

In [10]:
readout = esp.nn.readout.janossy.JanossyPooling(
    in_features=128, config=[128, "relu", 128, "relu", 128, "relu"],
    out_features={                  # define modular MM parameters Espaloma will assign
        1: {"e": 1, "s": 1},        # atom hardness and electronegativity
        2: {"log_coefficients": 2}, # bond linear combination, enforce positive
        3: {"log_coefficients": 2}, # angle linear combination, enforce positive
        4: {"k": 6},                # torsion barrier heights (can be positive or negative)
    },
)

espaloma_model = torch.nn.Sequential(
                 representation, readout, esp.nn.readout.janossy.ExpCoefficients(),
                 esp.mm.geometry.GeometryInGraph(),
                 esp.mm.energy.EnergyInGraph(),
)

In [11]:
if torch.cuda.is_available():
    espaloma_model = espaloma_model.cuda()

In [12]:
loss_fn = esp.metrics.GraphMetric(
        base_metric=torch.nn.MSELoss(), # use mean-squared error loss
        between=['u', "u_ref"],         # between predicted and QM energies
        level="g",                      # compare on graph level
)

In [13]:
epochs = 5
optimizer = torch.optim.Adam(espaloma_model.parameters(), 1e-4)

In [14]:
for epoch in range(epochs):
    for g in ds_tr:
        optimizer.zero_grad()
        if torch.cuda.is_available():
            g.heterograph = g.heterograph.to("cuda:0")
        g = espaloma_model(g.heterograph)
        loss = loss_fn(g)
        loss.backward()
        optimizer.step()
        torch.save(espaloma_model.state_dict(), "%s.th" % epoch)

torch.save(espaloma_model.state_dict(), "espaloma_model.pt")

In [15]:
inspect_metric = esp.metrics.center(torch.nn.L1Loss()) # use mean-squared error loss

In [16]:
loss_tr = []
loss_vl = []

In [17]:
with torch.no_grad():
    for epoch in range(epochs):
        espaloma_model.load_state_dict(
            torch.load("%s.th" % epoch)
        )

        # training set performance
        u = []
        u_ref = []
        for g in ds_tr:
            if torch.cuda.is_available():
                g.heterograph = g.heterograph.to("cuda:0")
            espaloma_model(g.heterograph)
            u.append(g.nodes['g'].data['u'])
            u_ref.append(g.nodes['g'])
        u = torch.cat(u, dim=0)
        u_ref = torch.cat(u_ref, dim=0)
        loss_tr.append(inspect_metric(u, u_ref))


        # validation set performance
        u = []
        u_ref = []
        for g in ds_vl:
            if torch.cuda.is_available():
                g.heterograph = g.heterograph.to("cuda:0")
            espaloma_model(g.heterograph)
            u.append(g.nodes['g'].data['u'])
            u_ref.append(g.nodes['g'])
        u = torch.cat(u, dim=0)
        u_ref = torch.cat(u_ref, dim=0)
        loss_vl.append(inspect_metric(u, u_ref))

TypeError: expected Tensor as element 0 in argument 0, but got NodeSpace

In [ ]:
loss_tr = np.array(loss_tr) * 627.5
loss_vl = np.array(loss_vl) * 627.5

In [ ]:
plt.plot(loss_tr, label="train")
plt.plot(loss_vl, label="valid")
plt.yscale("log")
plt.legend()